# Census Data Extraction Pipeline
Focuses on U.S. Census Bureau economic indicators used in the Reliance Inc. analytics platform.

In [1]:

# Import required libraries
import requests
import pandas as pd
from datetime import datetime
import os
from pathlib import Path
import time
from IPython.display import display
from urllib.parse import urlencode
import warnings
warnings.filterwarnings('ignore')

print("✓ Libraries ready for Census extraction")


✓ Libraries ready for Census extraction


In [2]:

# Configuration
API_KEYS = {
    'CENSUS': '87f911d32c71d4325b27bea0d9d72358be85825c',
}

START_DATE = '2015-01-01'
END_DATE = datetime.now().strftime('%Y-%m-%d')

OUTPUT_DIR = Path('extracted_data/census')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

AUDIT_LOG = []

print(f"✓ Configuration set | Range: {START_DATE} → {END_DATE}")
print(f"  Output directory: {OUTPUT_DIR.resolve()}")


✓ Configuration set | Range: 2015-01-01 → 2025-10-25
  Output directory: /Users/matthewtonks/Repositories/BUS 659/Class Folder/Final Project/Project 2/extracted_data/census


In [3]:

def log_extraction(source, status, records=0, error_msg=None):
    AUDIT_LOG.append({
        'timestamp': datetime.now(),
        'source': source,
        'status': status,
        'records': records,
        'error': error_msg
    })

def retry_request(url, params=None, max_retries=3, backoff_factor=2):
    for attempt in range(max_retries):
        try:
            response = requests.get(url, params=params, timeout=30)
            response.raise_for_status()
            return response
        except requests.exceptions.RequestException as exc:
            if attempt == max_retries - 1:
                raise
            wait = backoff_factor ** attempt
            print(f"  Retry attempt {attempt + 1} after {wait}s...")
            time.sleep(wait)

def save_data(df, filename, format='csv'):
    path = OUTPUT_DIR / filename
    if format == 'csv':
        df.to_csv(path, index=False)
    elif format == 'parquet':
        df.to_parquet(path, index=False)
    elif format == 'json':
        df.to_json(path, orient='records', date_format='iso')
    print(f"  ✓ Saved to {path}")
    return path

print("✓ Helper utilities ready")


✓ Helper utilities ready


## Construction Spending (VIP)

In [5]:

def extract_census_construction(api_key, start_date=START_DATE):
    base_url = "https://api.census.gov/data/timeseries/eits/vip"
    start_ym = pd.to_datetime(start_date).strftime('%Y-%m')
    params = {
        'get': 'cell_value,time_slot_id,category_code,data_type_code,seasonally_adj',
        'for': '',
        'time': f'from {start_ym}',
        'key': api_key
    }
    request_url = f"{base_url}?{urlencode(params, doseq=True)}"
    try:
        print(f"Extracting Census Construction (VIP)")
        print(f"  Request URL: {request_url}")
        response = retry_request(request_url)
        data = response.json()
        df = pd.DataFrame(data[1:], columns=data[0])
        df['date'] = pd.to_datetime(df['time_slot_id'], format='%Y%m', errors='coerce')
        df['value_millions'] = pd.to_numeric(df['cell_value'], errors='coerce')
        df['source'] = 'CENSUS_VIP'
        df['extraction_timestamp'] = datetime.now()
        if 'seasonally_adj' in df.columns:
            df['seasonally_adjusted'] = df['seasonally_adj'] == 'yes'
        df = df[['date', 'category_code', 'data_type_code', 'value_millions',
                    'seasonally_adjusted', 'source', 'extraction_timestamp']]
        df = df.dropna(subset=['value_millions'])
        print(f"  ✓ {len(df)} rows | {df['date'].min()} → {df['date'].max()}")
        log_extraction('CENSUS_VIP', 'SUCCESS', len(df))
        return df
    except Exception as err:
        print(f"  ✗ VIP extraction failed: {err}")
        log_extraction('CENSUS_VIP', 'FAILED', 0, str(err))
        return None


In [6]:
census_construction = extract_census_construction(API_KEYS['CENSUS'])
if census_construction is not None:
    save_data(census_construction, 'census_construction.csv')
census_construction.head() if census_construction is not None else None


Extracting Census Construction (VIP)
  Request URL: https://api.census.gov/data/timeseries/eits/vip?get=cell_value%2Ctime_slot_id%2Ccategory_code%2Cdata_type_code%2Cseasonally_adj&for=&time=from+2015-01&key=87f911d32c71d4325b27bea0d9d72358be85825c
  ✓ 36576 rows | NaT → NaT
  ✓ Saved to extracted_data/census/census_construction.csv


,date,category_code,data_type_code,value_millions,seasonally_adjusted,source,extraction_timestamp
0,NaT,A07XX,P,8603.0,True,CENSUS_VIP,2025-10-25 20:50:04.108764
1,NaT,20IX,MPCT,1.8,True,CENSUS_VIP,2025-10-25 20:50:04.108764
2,NaT,20IX,MPCT,-1.5,True,CENSUS_VIP,2025-10-25 20:50:04.108764
3,NaT,20IX,E_MPCV,0.4,False,CENSUS_VIP,2025-10-25 20:50:04.108764
4,NaT,20IX,E_MPCV,0.7,False,CENSUS_VIP,2025-10-25 20:50:04.108764


## Manufacturers' Shipments (M3)

In [7]:
def extract_census_m3(api_key, start_date=START_DATE, category_codes=None):
    """
    Extract Census M3 (Manufacturers' Shipments, Inventories, and Orders) data.
    Optionally filter to specific category codes after retrieval.
    """
    base_url = "https://api.census.gov/data/timeseries/eits/m3"
    start_ym = pd.to_datetime(start_date).strftime('%Y-%m')
    params = {
        'get': 'cell_value,time_slot_id,data_type_code,category_code,seasonally_adj',
        'for': 'US',
        'time': f'from {start_ym}',
        'key': api_key
    }
    request_url = f"{base_url}?{urlencode(params, doseq=True)}"
    try:
        print(f"\nExtracting Census M3 (Primary Metals focus)")
        print(f"  Request URL: {request_url}")
        response = retry_request(request_url)
        data = response.json()
        df = pd.DataFrame(data[1:], columns=data[0])
        if category_codes:
            df = df[df['category_code'].isin(category_codes)]
        df['date'] = pd.to_datetime(df['time_slot_id'], format='%Y%m', errors='coerce')
        df['value_millions'] = pd.to_numeric(df['cell_value'], errors='coerce')
        df['naics_code'] = df['category_code']
        df['indicator'] = df['data_type_code']
        df['source'] = 'CENSUS_M3'
        df['extraction_timestamp'] = datetime.now()
        data_type_map = {
            'SM': 'Shipments Monthly',
            'NO': 'New Orders',
            'UO': 'Unfilled Orders',
            'TI': 'Total Inventories',
            'MI': 'Materials and Supplies Inventory',
            'WI': 'Work-in-Process Inventory',
            'FI': 'Finished Goods Inventory'
        }
        df['indicator_name'] = df['indicator'].map(data_type_map).fillna(df['indicator'])

        if 'seasonally_adj' in df.columns:
            df['seasonally_adjusted'] = df['seasonally_adj'] == 'yes'
        df = df[['date', 'naics_code', 'indicator', 'indicator_name', 'value_millions',
                 'seasonally_adjusted', 'source', 'extraction_timestamp']]
        df = df.dropna(subset=['value_millions'])
        print(f"  ✓ {len(df)} rows | {df['date'].min()} → {df['date'].max()}")
        log_extraction('CENSUS_M3', 'SUCCESS', len(df))
        return df
    except Exception as err:
        print(f"  ✗ M3 extraction failed: {err}")
        log_extraction('CENSUS_M3', 'FAILED', 0, str(err))
        return None


In [8]:
census_m3 = extract_census_m3(API_KEYS['CENSUS'])
if census_m3 is not None:
    save_data(census_m3, 'census_m3_primary_metals.csv')
census_m3.head() if census_m3 is not None else None



Extracting Census M3 (Primary Metals focus)
  Request URL: https://api.census.gov/data/timeseries/eits/m3?get=cell_value%2Ctime_slot_id%2Cdata_type_code%2Ccategory_code%2Cseasonally_adj&for=US&time=from+2015-01&key=87f911d32c71d4325b27bea0d9d72358be85825c
  ✓ 184150 rows | NaT → NaT
  ✓ Saved to extracted_data/census/census_m3_primary_metals.csv


,date,naics_code,indicator,indicator_name,value_millions,seasonally_adjusted,source,extraction_timestamp
0,NaT,32S,MPCUO,MPCUO,-2.4,False,CENSUS_M3,2025-10-25 20:50:31.497447
1,NaT,32S,MPCUO,MPCUO,-2.1,False,CENSUS_M3,2025-10-25 20:50:31.497447
2,NaT,32S,MPCUO,MPCUO,-1.1,False,CENSUS_M3,2025-10-25 20:50:31.497447
3,NaT,32S,MPCUO,MPCUO,-2.5,True,CENSUS_M3,2025-10-25 20:50:31.497447
4,NaT,32S,MPCUO,MPCUO,-1.6,True,CENSUS_M3,2025-10-25 20:50:31.497447


## Residential Construction (RES)

In [ ]:

                def extract_census_res(api_key, start_date=START_DATE):
                    base_url = "https://api.census.gov/data/timeseries/eits/resconst"
                    start_ym = pd.to_datetime(start_date).strftime('%Y-%m')
                    params = {
                        'get': 'cell_value,time_slot_id,category_code,data_type_code,seasonally_adj',
                        'for': '',
                        'time': f'from {start_ym}',
                        'key': api_key
                    }
                    request_url = f"{base_url}?{urlencode(params, doseq=True)}"
                    try:
                        print(f"
Extracting Census RES (Residential Construction)")
                        print(f"  Request URL: {request_url}")
                        response = retry_request(request_url)
                        data = response.json()
                        df = pd.DataFrame(data[1:], columns=data[0])
                        df['date'] = pd.to_datetime(df['time_slot_id'], format='%Y%m', errors='coerce')
                        df['value_units'] = pd.to_numeric(df['cell_value'], errors='coerce')
                        df['source'] = 'CENSUS_RES'
                        df['extraction_timestamp'] = datetime.now()
                        if 'seasonally_adj' in df.columns:
                            df['seasonally_adjusted'] = df['seasonally_adj'] == 'yes'
                        df = df[['date', 'category_code', 'data_type_code', 'value_units',
                                 'seasonally_adjusted', 'source', 'extraction_timestamp']]
                        df = df.dropna(subset=['value_units'])
                        print(f"  ✓ {len(df)} rows | {df['date'].min()} → {df['date'].max()}")
                        log_extraction('CENSUS_RES', 'SUCCESS', len(df))
                        return df
                    except Exception as err:
                        print(f"  ✗ RES extraction failed: {err}")
                        log_extraction('CENSUS_RES', 'FAILED', 0, str(err))
                        return None


In [ ]:
census_res = extract_census_res(API_KEYS['CENSUS'])
if census_res is not None:
    save_data(census_res, 'census_res.csv')
census_res.head() if census_res is not None else None


## Monthly Retail Trade (MRTS)

In [ ]:

                def extract_census_mrts(api_key, start_date=START_DATE, categories=None):
                    base_url = "https://api.census.gov/data/timeseries/eits/mrts"
                    start_ym = pd.to_datetime(start_date).strftime('%Y-%m')
                    if categories is None:
                        categories = ['441', '444', '4441']
                    frames = []
                    for category in categories:
                        params = {
                            'get': 'cell_value,time_slot_id,data_type_code,category_code,seasonally_adj',
                            'for': 'US',
                            'time': f'from {start_ym}',
                            'category_code': category,
                            'key': api_key
                        }
                        request_url = f"{base_url}?{urlencode(params, doseq=True)}"
                        try:
                            print(f"
  Extracting MRTS category {category}")
                            print(f"    URL: {request_url}")
                            response = retry_request(request_url)
                            data = response.json()
                            frames.append(pd.DataFrame(data[1:], columns=data[0]))
                        except Exception as err:
                            print(f"    ✗ Category {category} failed: {err}")
                    if not frames:
                        log_extraction('CENSUS_MRTS', 'FAILED', 0, 'All category requests failed')
                        return None
                    df = pd.concat(frames, ignore_index=True)
                    df['date'] = pd.to_datetime(df['time_slot_id'], format='%Y%m', errors='coerce')
                    df['value_millions'] = pd.to_numeric(df['cell_value'], errors='coerce')
                    df['source'] = 'CENSUS_MRTS'
                    df['extraction_timestamp'] = datetime.now()
                    category_names = {
                        '441': 'Motor Vehicle and Parts Dealers',
                        '444': 'Building Material & Garden Equipment',
                        '4441': 'Building Material and Supplies Dealers'
                    }
                    df['category_name'] = df['category_code'].map(category_names).fillna(df['category_code'])
                    if 'seasonally_adj' in df.columns:
                        df['seasonally_adjusted'] = df['seasonally_adj'] == 'yes'
                    df = df[['date', 'category_code', 'category_name', 'value_millions',
                             'seasonally_adjusted', 'source', 'extraction_timestamp']]
                    df = df.dropna(subset=['value_millions'])
                    print(f"
✓ MRTS total rows: {len(df)}")
                    log_extraction('CENSUS_MRTS', 'SUCCESS', len(df))
                    return df


In [ ]:
census_mrts = extract_census_mrts(API_KEYS['CENSUS'])
if census_mrts is not None:
    save_data(census_mrts, 'census_mrts_selected.csv')
census_mrts.head() if census_mrts is not None else None


## Advance Durable Goods (ADVM3)

In [ ]:

                def extract_census_advm3(api_key, start_date=START_DATE):
                    base_url = "https://api.census.gov/data/timeseries/eits/advm3"
                    start_ym = pd.to_datetime(start_date).strftime('%Y-%m')
                    params = {
                        'get': 'cell_value,time_slot_id,data_type_code,category_code,seasonally_adj',
                        'for': 'US',
                        'time': f'from {start_ym}',
                        'key': api_key
                    }
                    request_url = f"{base_url}?{urlencode(params, doseq=True)}"
                    try:
                        print(f"
Extracting Census ADVM3 (Advance Durable Goods)")
                        print(f"  Request URL: {request_url}")
                        response = retry_request(request_url)
                        data = response.json()
                        df = pd.DataFrame(data[1:], columns=data[0])
                        df['date'] = pd.to_datetime(df['time_slot_id'], format='%Y%m', errors='coerce')
                        df['value_millions'] = pd.to_numeric(df['cell_value'], errors='coerce')
                        df['indicator'] = df['data_type_code']
                        df['source'] = 'CENSUS_ADVM3'
                        df['is_advance_release'] = True
                        df['extraction_timestamp'] = datetime.now()
                        data_type_map = {
                            'SM': 'Shipments Monthly',
                            'NO': 'New Orders',
                            'UO': 'Unfilled Orders'
                        }
                        df['indicator_name'] = df['indicator'].map(data_type_map).fillna(df['indicator'])
                        if 'seasonally_adj' in df.columns:
                            df['seasonally_adjusted'] = df['seasonally_adj'] == 'yes'
                        df = df[['date', 'category_code', 'indicator', 'indicator_name', 'value_millions',
                                 'seasonally_adjusted', 'is_advance_release', 'source', 'extraction_timestamp']]
                        df = df.dropna(subset=['value_millions'])
                        print(f"  ✓ {len(df)} rows | {df['date'].min()} → {df['date'].max()}")
                        log_extraction('CENSUS_ADVM3', 'SUCCESS', len(df))
                        return df
                    except Exception as err:
                        print(f"  ✗ ADVM3 extraction failed: {err}")
                        log_extraction('CENSUS_ADVM3', 'FAILED', 0, str(err))
                        return None


In [ ]:
census_advm3 = extract_census_advm3(API_KEYS['CENSUS'])
if census_advm3 is not None:
    save_data(census_advm3, 'census_advm3.csv')
census_advm3.head() if census_advm3 is not None else None


## Optional: Manual Steel Imports (FT900)

In [ ]:

                def process_census_steel_imports(filepath):
                    try:
                        print(f"
Processing Census Steel Imports file: {filepath}")
                        if str(filepath).endswith('.csv'):
                            df = pd.read_csv(filepath)
                        elif str(filepath).endswith(('.xls', '.xlsx')):
                            df = pd.read_excel(filepath)
                        else:
                            raise ValueError("Unsupported file format")
                        df['source'] = 'CENSUS_FT900'
                        df['extraction_timestamp'] = datetime.now()
                        if 'value_thousands' in df.columns and 'quantity_metric_tons' in df.columns:
                            df['unit_price'] = (df['value_thousands'] * 1000) / df['quantity_metric_tons']
                        log_extraction('CENSUS_FT900', 'SUCCESS', len(df))
                        return df
                    except Exception as err:
                        print(f"  ✗ Steel imports processing failed: {err}")
                        log_extraction('CENSUS_FT900', 'FAILED', 0, str(err))
                        return None


## Audit Summary

In [ ]:

if AUDIT_LOG:
    audit_df = pd.DataFrame(AUDIT_LOG)
    summary = audit_df.groupby(['source', 'status']).agg({'records': 'sum'}).reset_index()
    display(summary)
    save_data(audit_df, 'audit_log.csv')
else:
    print("No extraction attempts logged yet.")
